# 🏭 Processes

*Section 14.2 of the notes.*

In notebook `01-threads` we saw that threads cannot speed up Python
arithmetic: there is one interpreter lock 🔒, and only one thread at a time may
hold it.

A **process** is a separate running program, with its **own interpreter**, its
**own lock**, and its **own memory**. Several processes therefore really run
Python code at the same time, one per core. 🎉

Nothing comes for free, though. Processes cost something that threads do not:

* 🐣 **starting** them takes time: each one starts a fresh Python and imports
  the modules it needs;
* 📦 **sending data** between them takes time: they do not share memory, so
  every argument and every result is converted to bytes, copied over, and
  rebuilt on the other side.

This notebook measures the gain and the two costs, so that you can tell when
processes are worth it.

## ⚙️ Setup

A `ProcessPoolExecutor` works like the `ThreadPoolExecutor` of the previous
notebook: a pool of workers, `map` to hand out jobs, and a `with` block that
waits for all of them. The workers are now processes.

📌 **Why are the functions in `tasks.py`?** A new worker process starts with
an empty interpreter. To run a job, it receives the *name* of the function,
for example `tasks.cpu_work`, imports the module `tasks`, and looks the
function up. A function defined in a notebook cell has no module to be
imported from, so the workers could not find it. (In a script, the same
reason requires the code that starts the pool to sit under
`if __name__ == "__main__":`, which the notes explain in Section 14.2.)

⏱️ Timings use `%timeit -o -q -r 3 -n 1` and `.best`, as in the previous
notebook.

🖥️ **Spawn, the same on every computer.** Python can start a worker process in
several ways, and the default depends on the operating system. We ask for
*spawn*, where every worker is a fresh Python, with `mp_context=CTX` in every
pool, so that the notebook behaves the same on every computer. The last cell of
the notebook compares the ways.

In [ ]:
import math
import multiprocessing
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np

import tasks

CTX = multiprocessing.get_context("spawn")  # start workers as fresh Pythons

print("cores:", os.cpu_count())

## 1. 🚀 Speed-up on independent tasks

We run `N` copies of `tasks.cpu_work(n)`, the pure Python loop that threads
could not speed up, with `N` equal to the number of cores. First one after the
other, then with pools of 2, 4, and `N` worker processes.

How fast can it possibly be? Each worker does one task at a time. With `k`
workers the `N` tasks are done in **rounds** of `k` tasks, and the last round
may be only partly full. So they need `ceil(N / k)` rounds, and the best
possible speed-up is

$$ \frac{N}{\lceil N/k \rceil} . $$

For example, 18 tasks on 4 workers need 5 rounds (4 + 4 + 4 + 4 + 2), so the
best possible speed-up is 18 / 5 = 3.6, not 4. The cell prints this bound next
to the measured speed-up.

🤔 *Guess: will the measured speed-up reach the bound?* (The cell takes about
a minute.)

In [ ]:
N = os.cpu_count()  # number of jobs
n = 20_000_000  # loop steps per job

t_seq = %timeit -o -q -r 3 -n 1 [tasks.cpu_work(n) for _ in range(N)]
t_seq = t_seq.best
print(f"sequential   {t_seq:5.2f} s")


def with_processes(k):
    with ProcessPoolExecutor(max_workers=k, mp_context=CTX) as pool:
        list(pool.map(tasks.cpu_work, [n] * N))  # one argument per job: N copies of n


for k in (2, 4, N):
    t = %timeit -o -q -r 3 -n 1 with_processes(k)
    bound = N / math.ceil(N / k)  # best possible speed-up with k workers
    print(
        f"{k:2d} processes {t.best:5.2f} s   speed-up {t_seq / t.best:5.2f}x"
        f"   (best possible {bound:4.1f}x)"
    )

💡 **What happened?** Unlike threads, processes give a real speed-up on pure
Python work. 🎉 With few workers it is close to the bound. It can even land slightly
above it: timings vary by a few per cent from run to run, and the one-process
time that every speed-up is divided by varies too. With all cores
it falls further below the bound, for two reasons:

* starting the pool costs time, measured in the next section;
* not all cores are equally fast. Many recent processors, from Intel and
  Apple among others, mix fast *performance* cores with slower *efficiency*
  cores, and a round ends only when its slowest worker finishes.

👉 For Python code that keeps the processor busy, **processes** are the tool,
not threads.

## 2. 🐣 What the pool costs before any work

How long does it take to start a pool of `N` workers? To make sure that
every worker really starts, we give each of them a job that only waits for
0.5 s (Python starts workers only when there is work for them). Everything
above 0.5 s is the cost of starting the workers: a fresh Python each, which
imports `tasks` and NumPy.

In [ ]:
def start_pool_and_wait():
    with ProcessPoolExecutor(max_workers=N, mp_context=CTX) as pool:
        list(pool.map(tasks.wait, [0.5] * N))


t_pool = %timeit -o -q -r 3 -n 1 start_pool_and_wait()
t_task = %timeit -o -q -r 3 -n 1 tasks.cpu_work(n)
print(f"starting a pool of {N} workers: {t_pool.best - 0.5:.2f} s")
print(f"one task of tasks.cpu_work:    {t_task.best:.2f} s")

💡 **What happened?** Starting the pool costs a fraction of a second. That
is small next to one `cpu_work` task, but huge next to a task that takes a
millisecond.

👉 **Rule of thumb:** each task should take much longer than the overhead of
handing it to a worker.

🔥 **A warm pool for the timings.** From now on we want to time the work and
the data transfer, not the start-up. `tasks.warm_pool(N)` starts a pool of
`N` workers and first gives each of them a job of 0.1 s, so that the pool has
to start all of them. When it returns, every worker is running and idle.

What if you have many small tasks? Every task travels to a worker as a
message 📨, and its result travels back as another message. With 2000 tiny
tasks that is 4000 messages. The option `chunksize=100` packs 100 tasks into
one message, so only 20 messages go out. The cell below uses one warm pool for
both timings.

🤔 *Guess: how much does `chunksize` help?*

In [ ]:
small = [10_000] * 2_000  # 2000 tasks of 10 000 loop steps each

with tasks.warm_pool(N) as pool:
    t1 = %timeit -o -q -r 3 -n 1 list(pool.map(tasks.cpu_work, small))
    t2 = %timeit -o -q -r 3 -n 1 list(pool.map(tasks.cpu_work, small, chunksize=100))

print(f"2000 short tasks, one by one     {t1.best:5.2f} s")
print(f"2000 short tasks, chunksize=100  {t2.best:5.2f} s")

💡 **What happened?** Grouping the tasks is a few times faster, because
most of the time of the first version went into sending messages, not into the
work.

## 3. 📦 Sending data costs as well

Each worker has its own memory, so an array passed to a task must be
**copied** into the worker: it is converted to bytes (this is called
*pickling* 🥒), sent through a pipe, and rebuilt on the other side.

We split an array of 20 million numbers (160 MB) into `N` pieces, one per
worker, and compare two jobs on the pieces:

* `np.sum`, which is **cheap**: one quick pass over the numbers;
* `tasks.harmonics_sent`, which is **expensive**: it adds up
  `sin(x) + sin(2x) + ... + sin(5x)` for every entry.

Both receive the same data, so the cost of sending is the same. Only the
amount of work per byte differs. The pool is warm, so only the sending and the
work are timed.

🤔 *Guess: for which job is the pool faster than working in this process?*

In [ ]:
rng = np.random.default_rng(0)
big = rng.standard_normal(20_000_000)  # 160 MB
pieces = np.array_split(big, N)  # N pieces, one per worker

print(f"{'job':22s} {'in this process':>16s} {'in the pool':>12s}")
for name, job in [
    ("np.sum (cheap)", np.sum),
    ("harmonics (expensive)", tasks.harmonics_sent),
]:
    t_here = %timeit -o -q -r 3 -n 1 [job(p) for p in pieces]
    with tasks.warm_pool(N) as pool:
        t_pool = %timeit -o -q -r 3 -n 1 list(pool.map(job, pieces))

    ratio = t_here.best / t_pool.best
    if ratio > 1:
        verdict = f"pool {ratio:.0f}x faster"
    else:
        verdict = f"pool {1 / ratio:.0f}x slower"

    print(f"{name:22s} {t_here.best:14.3f} s {t_pool.best:10.3f} s   {verdict}")

💡 **What happened?**

* For the **cheap** job the pool is *slower* than doing the work right here:
  copying 160 MB to the workers takes longer than the sum itself. 🐌
* For the **expensive** job the pool is *faster*: the work is spread over
  all cores and is much larger than the cost of copying the data. 🏎️

👉 Processes pay off when there is a lot of **work per byte sent**. The next
notebook, `03-shared-memory`, removes the copy altogether: the workers read
the array from a block of memory that all of them share.

## 4. 📊 Progress while the work runs

As with threads, `submit` returns a **future** for each task, and
`as_completed` hands the futures back as they finish. Here 8 tasks run on 4
workers, and a line is printed every time one of them is done. This is handy
for long computations, where you want to see that something is happening. ⏳

In [ ]:
with ProcessPoolExecutor(max_workers=4, mp_context=CTX) as pool:
    futures = [pool.submit(tasks.cpu_work, 2_000_000) for _ in range(8)]
    for i, f in enumerate(as_completed(futures), 1):
        f.result()  # the value itself; here we only count the finished tasks

        print(f"{i}/{len(futures)} done", end="  ")

💡 The 4 workers take the first 4 tasks. As each of them finishes, it picks
up one of the remaining 4, so the tasks finish in two waves of four.

## 🎯 Summary

| | Threads | Processes |
|---|---|---|
| Python arithmetic | ❌ no gain (one lock) | ✅ real speed-up |
| Starting | ✅ almost free | ❌ a fraction of a second |
| Passing data | ✅ shared memory, free | ❌ copied, costs time |
| Best for | waiting, compiled code | long Python tasks with little data |

## 🧪 Try it yourself

1. In Section 1, add `k = 3` and `k = 6` to the loop. Do the measured
   speed-ups follow the bound `N / ceil(N / k)`?
2. In Section 2, try `chunksize=10` and `chunksize=1000`. Which is fastest,
   and why can a very large chunk be worse? (Hint: how many workers get work
   when there are only 2 chunks?)
3. In Section 3, change the number of terms in `tasks.harmonics` (the
   argument `terms`) and find the amount of work at which the pool starts to
   win.

## 🔍 Going further: spawn, fork, and forkserver

Python has three ways of starting a worker process, called *start methods*:

* **spawn** 🐣: the worker is a **fresh Python**. It imports the modules it
  needs, such as `tasks` and NumPy, and receives the function to run by its
  name.
* **fork** 🧬: the worker is a **copy of the running program**, made by the
  operating system in one step. It starts with everything the parent had: the
  imported modules, the variables, even the functions defined in notebook
  cells.
* **forkserver** 🏗️: at the start, Python forks one small, clean *server*
  process, which has not run any of your code yet. Every worker is then forked
  from this server, never from your program.

| | spawn | fork | forkserver |
|---|---|---|---|
| a worker is | a fresh Python | a copy of the running program | a copy of a clean server |
| starting a worker | slower: imports everything again | fast | in between |
| functions defined in a notebook | ❌ not found | ✅ work | ❌ not found |

Which of them exist depends on the operating system:

| | spawn | fork | forkserver |
|---|---|---|---|
| Linux | ✅ | ✅ default up to Python 3.13 | ✅ default from Python 3.14 |
| macOS | ✅ default | ✅ but unsafe | ✅ |
| Windows | ✅ default | ❌ Windows has no fork | ❌ |

**Why is fork unsafe?** ⚠️ A program often runs threads that you never
started yourself, for example the threads of a linear algebra library
(notebook `03-shared-memory`, Section 5). Fork copies the memory of the program
but only the one thread that called it. Suppose another thread was holding a
lock at that moment. In the copy, the lock is still taken, but the thread that
would release it does not exist, so the worker waits for it forever and hangs.
🥶 On macOS, Apple's own system libraries run such threads, and this is why
Python switched the default on macOS to spawn in version 3.8. Python 3.12 and
later print a warning when a program with several threads forks, and Python
3.14 moved the Linux default to forkserver.

👉 **What this means for you.** spawn works the same everywhere, which is why
this notebook uses it. Two habits follow from it, and they keep your code
correct with every start method:

1. put the functions that workers run in a module, such as `tasks.py`;
2. in a script, start the pool under `if __name__ == "__main__":`.